# Pending model evaluation (pre-merge QA)

Compare **canonical models** present under the current Random-vs-LLM and Dragon-vs-LLM log trees with **`elo_refined.csv`**, check **metadata** coverage for models not yet on the leaderboard, and summarize **game health** (wins / draws / losses, interruptions, termination reasons) per model and per run folder.

**Log scope:** only `_logs/rand_vs_llm` and `_logs/engine_vs_llm` (not legacy `_pre_aug_2025` trees). This notebook’s “models in logs” set is **narrower** than a full Elo regeneration via `get_refined_csv.py`, which still ingests legacy directories.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# Project root: parent of this notebook's directory (analysis/)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from data_processing.get_refined_csv import (
    load_game_logs,
    load_game_log,
    GameMode,
    ALIASES,
    FILTER_OUT_MODELS,
    FILTER_OUT_PATH_KEYWORDS,
    MODEL_OVERRIDES,
    DATE_AFTER,
    _model_label_from_run_json,
    _white_opponent_from_run_dir,
    SUPPRESS_MODEL_RECOVERY_WARNINGS,
)
RAND_LOGS_ONLY = [os.path.join(project_root, "_logs/rand_vs_llm")]
ENGINE_LOGS_ONLY = [os.path.join(project_root, "_logs/engine_vs_llm")]

ELO_CSV = os.path.join(project_root, "data_processing/elo_refined.csv")
MODELS_METADATA_CSV = os.path.join(project_root, "data_processing/models_metadata.csv")

print("project_root:", project_root)
print("DATE_AFTER (module):", DATE_AFTER)

project_root: /home/user/src/llm_chess
DATE_AFTER (module): None


<Figure size 640x480 with 0 Axes>

## Current `elo_refined.csv` snapshot

In [2]:
elo_df = pd.read_csv(ELO_CSV)
print("rows:", len(elo_df))
key_cols = [
    "Player",
    "total_games",
    "elo",
    "games_vs_random",
    "games_vs_dragon",
]
print("key columns present:", [c for c in key_cols if c in elo_df.columns])
elo_players = set(elo_df["Player"].dropna().astype(str).str.strip())
print("unique Player:", len(elo_players))

rows: 157
key columns present: ['Player', 'total_games', 'elo', 'games_vs_random', 'games_vs_dragon']
unique Player: 157


## Models present in logs (canonical labels)

Uses `load_game_logs` with the same `GameMode` and label rules as `build_refined_rows_from_logs` (`player_black.model` after ingestion, then `ALIASES`).

In [3]:
logs_random = load_game_logs(RAND_LOGS_ONLY, MODEL_OVERRIDES, mode=GameMode.RANDOM_VS_LLM)
logs_dragon = load_game_logs(ENGINE_LOGS_ONLY, MODEL_OVERRIDES, mode=GameMode.DRAGON_VS_LLM)


def canonical(log) -> str:
    m = log.player_black.model
    return ALIASES.get(m, m)


log_models_random = {canonical(log) for log in logs_random}
log_models_dragon = {canonical(log) for log in logs_dragon}
log_models = log_models_random | log_models_dragon

print("games loaded — random:", len(logs_random), " dragon:", len(logs_dragon))
print("unique canonical models (union):", len(log_models))
print("only in random:", len(log_models_random - log_models_dragon))
print("only in dragon:", len(log_models_dragon - log_models_random))

games loaded — random: 1980  dragon: 1777
unique canonical models (union): 84
only in random: 28
only in dragon: 11


## Diff: logs vs Elo CSV (`pending`)

In [4]:
pending = sorted(log_models - elo_players)
filter_set = set(FILTER_OUT_MODELS)
pending_also_filtered = sorted(set(pending) & filter_set)
pending_truly_new = sorted(set(pending) - filter_set)

print("overlap summary")
print("  elo_refined Player count:", len(elo_players))
print("  canonical models in scoped logs:", len(log_models))
print("  pending (in logs, not in elo_refined):", len(pending))
print()
print("pending ∩ FILTER_OUT_MODELS (explicitly filtered in pipeline):", len(pending_also_filtered))
if pending_also_filtered:
    print("  ", pending_also_filtered)
print("pending not in FILTER_OUT_MODELS (typically threshold / path / ingest skips):", len(pending_truly_new))
print()
print("pending (sorted):")
for p in pending:
    print(" ", p)

overlap summary
  elo_refined Player count: 157
  canonical models in scoped logs: 84
  pending (in logs, not in elo_refined): 28

pending ∩ FILTER_OUT_MODELS (explicitly filtered in pipeline): 7
   ['chess-4b-thinking-1218', 'cursor_cli_sonnet_4.5', 'gpt-5-codex-2025-09-15-low', 'gpt-5.1-codex-mini-2025-11-13-medium', 'gpt-oss:20b-low', 'llama-4-scout-17b-16e-instruct', 'ring-mini-2.0@q4_k_m']
pending not in FILTER_OUT_MODELS (typically threshold / path / ingest skips): 21

pending (sorted):
  DeepSeek-R1-0528
  DeepSeek-V3.2
  Kimi-K2.5
  anthropic.claude-opus-4-6-v1
  anthropic.claude-opus-4-6-v1-with-thinking
  anthropic.claude-sonnet-4-6
  anthropic.claude-sonnet-4-6-with-thinking
  chess-4b-thinking-1218
  composer_cli
  cursor_cli_sonnet_4.5
  gemini-3.1-flash-lite-preview
  gemini-3.1-pro-preview
  gpt-5-codex-2025-09-15-low
  gpt-5.1-codex-mini-2025-11-13-medium
  gpt-5.4-low
  gpt-5.4-medium
  gpt-5.4-mini-2026-03-17-high
  gpt-5.4-nano-2026-03-17-high
  gpt-oss:20b-low
  gro

## Metadata coverage for `pending` models

Same style as `timeline.ipynb`: exact match on `models_metadata.csv` column `model` (stripped).

In [5]:
metadata_df = pd.read_csv(MODELS_METADATA_CSV)
metadata_df["model"] = metadata_df["model"].astype(str).str.strip()
metadata_models = set(metadata_df["model"])

pending_missing_meta = sorted([p for p in pending if p not in metadata_models])
pending_with_meta = sorted([p for p in pending if p in metadata_models])

n = len(pending)
if n == 0:
    print("No pending models — metadata check skipped.")
else:
    print("=" * 80)
    print("METADATA COVERAGE (pending models only)")
    print("=" * 80)
    print(f"Total pending: {n}")
    print(f"With metadata:    {len(pending_with_meta)} ({100 * len(pending_with_meta) / n:.1f}%)")
    print(f"WITHOUT metadata: {len(pending_missing_meta)} ({100 * len(pending_missing_meta) / n:.1f}%)")
    print("=" * 80)
    print()
    for p in pending:
        mark = "✓" if p in metadata_models else "✗ MISSING"
        print(f"{mark} {p}")

METADATA COVERAGE (pending models only)
Total pending: 28
With metadata:    11 (39.3%)
WITHOUT metadata: 17 (60.7%)

✗ MISSING DeepSeek-R1-0528
✓ DeepSeek-V3.2
✓ Kimi-K2.5
✗ MISSING anthropic.claude-opus-4-6-v1
✓ anthropic.claude-opus-4-6-v1-with-thinking
✗ MISSING anthropic.claude-sonnet-4-6
✓ anthropic.claude-sonnet-4-6-with-thinking
✗ MISSING chess-4b-thinking-1218
✗ MISSING composer_cli
✗ MISSING cursor_cli_sonnet_4.5
✗ MISSING gemini-3.1-flash-lite-preview
✗ MISSING gemini-3.1-pro-preview
✓ gpt-5-codex-2025-09-15-low
✓ gpt-5.1-codex-mini-2025-11-13-medium
✗ MISSING gpt-5.4-low
✗ MISSING gpt-5.4-medium
✗ MISSING gpt-5.4-mini-2026-03-17-high
✗ MISSING gpt-5.4-nano-2026-03-17-high
✗ MISSING gpt-oss:20b-low
✓ grok-4-1-fast-non-reasoning
✓ grok-4-1-fast-reasoning
✓ grok-4-fast-non-reasoning
✓ grok-4-fast-reasoning
✓ llama-4-scout-17b-16e-instruct
✗ MISSING qwen-3-235b-a22b-thinking-2507-low
✗ MISSING qwen3.5-9b@q8_0
✗ MISSING ring-mini-2.0@q4_k_m
✗ MISSING zai-glm-4.7


## Per-model stats (pending models only)

Win / loss / draw definitions match `build_refined_rows_from_logs`: Black LLM wins if `winner in ("Player_Black", "NoN_Synthesizer")`, opponent wins if `winner == player_white.name`, else draw. **Interrupted** uses `GameLog.is_interrupted` (error, unknown issue, max turns, too many wrong actions).

In [6]:
combined_logs = logs_random + logs_dragon
pending_set = set(pending)
pending_logs = [log for log in combined_logs if canonical(log) in pending_set]


def outcome_wld(log):
    if log.winner in ("Player_Black", "NoN_Synthesizer"):
        return "wins"
    if log.winner == log.player_white.name:
        return "losses"
    return "draws"


rows = []
for log in pending_logs:
    rows.append(
        {
            "model": canonical(log),
            "outcome": outcome_wld(log),
            "interrupted": bool(log.is_interrupted),
            "reason": log.reason,
        }
    )

if not rows:
    stats_by_model = pd.DataFrame(
        columns=["model", "games", "wins", "losses", "draws", "interrupted"]
    )
    reason_ct = pd.DataFrame()
    print("No games for pending models.")
else:
    mini = pd.DataFrame(rows)
    agg = (
        mini.groupby("model", as_index=False)
        .agg(
            games=("outcome", "size"),
            wins=("outcome", lambda s: (s == "wins").sum()),
            losses=("outcome", lambda s: (s == "losses").sum()),
            draws=("outcome", lambda s: (s == "draws").sum()),
            interrupted=("interrupted", "sum"),
        )
        .sort_values(["model"])
        .reset_index(drop=True)
    )
    stats_by_model = agg
    reason_ct = pd.crosstab(mini["model"], mini["reason"])
    display(stats_by_model)
    print("Termination reasons (rows=models, columns=reason):")
    display(reason_ct)

,model,games,wins,losses,draws,interrupted
0,DeepSeek-R1-0528,50,6,1,43,33
1,DeepSeek-V3.2,29,6,5,18,1
2,Kimi-K2.5,31,11,6,14,7
3,anthropic.claude-opus-4-6-v1,41,0,41,0,41
4,anthropic.claude-opus-4-6-v1-with-thinking,25,5,2,18,3
5,anthropic.claude-sonnet-4-6,38,7,7,24,1
6,anthropic.claude-sonnet-4-6-with-thinking,15,0,5,10,1
7,chess-4b-thinking-1218,20,0,15,5,13
8,composer_cli,40,8,23,9,17
9,cursor_cli_sonnet_4.5,11,2,1,8,1


Termination reasons (rows=models, columns=reason):


reason,Checkmate,ERROR OCCURED,Fivefold repetition,Insufficient material,Max moves reached,Max turns in single dialog,Stalemate,Too many wrong actions
model,,,,,,,,
DeepSeek-R1-0528,7,33,0,2,7,0,1,0
DeepSeek-V3.2,11,1,0,1,15,0,1,0
Kimi-K2.5,15,5,0,5,2,0,2,2
anthropic.claude-opus-4-6-v1,0,0,0,0,0,41,0,0
anthropic.claude-opus-4-6-v1-with-thinking,7,3,0,8,6,0,1,0
anthropic.claude-sonnet-4-6,13,0,0,16,8,0,0,1
anthropic.claude-sonnet-4-6-with-thinking,4,0,1,4,4,1,1,0
chess-4b-thinking-1218,2,0,0,0,5,7,0,6
composer_cli,14,0,0,5,3,17,1,0


## Per run-folder stats (pending models)

`load_game_logs` does not attach file paths to `GameLog`. The walker below **duplicates** `load_game_logs` filtering and label logic for `RAND_LOGS_ONLY` and `ENGINE_LOGS_ONLY` only, and records `run_dir` as `os.path.dirname(file_path)`. **Keep in sync with** `data_processing/get_refined_csv.py::load_game_logs`.

In [7]:
def load_game_logs_with_paths(
    logs_dirs,
    model_overrides,
    mode: GameMode,
    project_root: str,
):
    """Mirror of load_game_logs; yields (GameLog, rel_run_dir)."""
    out = []
    if isinstance(logs_dirs, str):
        logs_dirs = [logs_dirs]
    processed_logs_dirs = []
    directory_aliases = {}
    for entry in logs_dirs:
        if isinstance(entry, dict):
            for dir_path, alias in entry.items():
                processed_logs_dirs.append(dir_path)
                directory_aliases[dir_path] = alias
        else:
            processed_logs_dirs.append(entry)

    for logs_dir in processed_logs_dirs:
        for root, _, files in os.walk(logs_dir):
            for file in files:
                if file.endswith(".json") and not file.endswith("_aggregate_results.json") and file != "_run.json":
                    file_path = os.path.join(root, file)
                    file_path_lower = file_path.lower()
                    if any(keyword.lower() in file_path_lower for keyword in FILTER_OUT_PATH_KEYWORDS):
                        continue
                    try:
                        game_log = load_game_log(file_path)
                        run_dir = os.path.dirname(file_path)
                        rel = os.path.relpath(run_dir, project_root)

                        if mode == GameMode.RANDOM_VS_LLM:
                            if game_log.player_white.name != "Random_Player":
                                continue
                            if game_log.player_black.name != "Player_Black":
                                continue
                            if logs_dir in directory_aliases:
                                model_name = directory_aliases[logs_dir]
                            else:
                                label_from_run = _model_label_from_run_json(run_dir)
                                model_name = label_from_run or game_log.player_black.model
                                if model_overrides:
                                    key = next((k for k in model_overrides if os.path.dirname(file_path).endswith(k)), None)
                                    if key:
                                        model_name = model_overrides[key]
                            game_log.player_black.model = model_name
                            out.append((game_log, rel))

                        elif mode == GameMode.DRAGON_VS_LLM:
                            if game_log.player_black.name != "Player_Black":
                                continue
                            is_new_format_base = os.path.normpath(logs_dir).endswith(
                                os.path.normpath("_logs/engine_vs_llm")
                            )
                            label_from_run = _model_label_from_run_json(run_dir)
                            model_name = label_from_run or game_log.player_black.model
                            if not is_new_format_base and model_name in ("placeholder", "Player_Black", "N/A", ""):
                                candidate = None
                                if game_log.usage_stats_black and game_log.usage_stats_black.details:
                                    candidate = next(
                                        (k for k in game_log.usage_stats_black.details.keys() if k != "total_cost"),
                                        None,
                                    )
                                if candidate:
                                    if not any(s in file_path for s in SUPPRESS_MODEL_RECOVERY_WARNINGS):
                                        print(
                                            f"WARNING: Recovered model '{candidate}' from usage_stats for log with invalid model '{model_name}' at {file_path}"
                                        )
                                    model_name = candidate
                                else:
                                    print(
                                        f"WARNING: Could not determine model ID for log at {file_path}. Player.model='{model_name}', usage_stats keys={list(game_log.usage_stats_black.details.keys()) if game_log.usage_stats_black.details else 'None'}"
                                    )
                            if label_from_run is None:
                                if is_new_format_base:
                                    print(
                                        f"WARNING: Missing or invalid _run.json for run at {run_dir}; skipping (engine_vs_llm new format)"
                                    )
                                    continue
                            if logs_dir in directory_aliases:
                                model_name = directory_aliases[logs_dir]
                            if model_overrides:
                                key = next((k for k in model_overrides if os.path.dirname(file_path).endswith(k)), None)
                                if key:
                                    model_name = model_overrides[key]
                            white_op = _white_opponent_from_run_dir(run_dir)
                            if white_op is None:
                                if is_new_format_base:
                                    print(f"WARNING: Could not infer dragon engine level for run at {run_dir}; skipping")
                                    continue
                                print(f"WARNING: Legacy path without parseable dragon level at {run_dir}; skipping")
                                continue
                            game_log.player_black.model = model_name
                            game_log.white_opponent = white_op
                            out.append((game_log, rel))
                    except Exception:
                        continue
    return out


path_random = load_game_logs_with_paths(
    RAND_LOGS_ONLY, MODEL_OVERRIDES, GameMode.RANDOM_VS_LLM, project_root
)
path_dragon = load_game_logs_with_paths(
    ENGINE_LOGS_ONLY, MODEL_OVERRIDES, GameMode.DRAGON_VS_LLM, project_root
)
path_all = path_random + path_dragon

path_rows = []
for log, rel in path_all:
    cm = canonical(log)
    if cm not in pending_set:
        continue
    path_rows.append(
        {
            "model": cm,
            "run_path": rel,
            "outcome": outcome_wld(log),
            "interrupted": bool(log.is_interrupted),
            "reason": log.reason,
            "white_opponent": getattr(log, "white_opponent", ""),
        }
    )

if not path_rows:
    stats_by_path = pd.DataFrame(
        columns=["model", "run_path", "games", "wins", "losses", "draws", "interrupted"]
    )
    print("No per-path rows for pending models.")
else:
    pm = pd.DataFrame(path_rows)
    stats_by_path = (
        pm.groupby(["model", "run_path"], as_index=False)
        .agg(
            games=("outcome", "size"),
            wins=("outcome", lambda s: (s == "wins").sum()),
            losses=("outcome", lambda s: (s == "losses").sum()),
            draws=("outcome", lambda s: (s == "draws").sum()),
            interrupted=("interrupted", "sum"),
        )
        .sort_values(["model", "run_path"])
        .reset_index(drop=True)
    )
    display(stats_by_path)
    reason_path = pd.crosstab([pm["model"], pm["run_path"]], pm["reason"])
    print("Reasons by (model, run_path):")
    display(reason_path)

,model,run_path,games,wins,losses,draws,interrupted
0,DeepSeek-R1-0528,_logs/engine_vs_llm/dragon-lvl-1/DeepSeek-R1-0...,10,0,0,10,5
1,DeepSeek-R1-0528,_logs/engine_vs_llm/dragon-lvl-2/DeepSeek-R1-0...,10,2,1,7,4
2,DeepSeek-R1-0528,_logs/rand_vs_llm/DeepSeek-R1-0528/2026-03-08-...,30,4,0,26,24
3,DeepSeek-V3.2,_logs/engine_vs_llm/dragon-lvl-1/DeepSeek-V3.2...,9,0,1,8,0
4,DeepSeek-V3.2,_logs/engine_vs_llm/dragon-lvl-2/DeepSeek-V3.2...,10,0,4,6,1
...,...,...,...,...,...,...,...
86,qwen-3-235b-a22b-thinking-2507-low,_logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a...,3,0,3,0,3
87,qwen-3-235b-a22b-thinking-2507-low,_logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a...,1,0,0,1,1
88,qwen3.5-9b@q8_0,_logs/rand_vs_llm/qwen-qwen3.5-9b/2026-03-03-1...,33,3,18,12,18
89,ring-mini-2.0@q4_k_m,_logs/rand_vs_llm/ring-mini-2.0-q4_k_m/2025-10...,33,0,33,0,33


Reasons by (model, run_path):


reason                                                                                 Checkmate  \
model                              run_path                                                        
DeepSeek-R1-0528                   _logs/engine_vs_llm/dragon-lvl-1/DeepSeek-R1-05...          0   
                                   _logs/engine_vs_llm/dragon-lvl-2/DeepSeek-R1-05...          3   
                                   _logs/rand_vs_llm/DeepSeek-R1-0528/2026-03-08-1...          4   
DeepSeek-V3.2                      _logs/engine_vs_llm/dragon-lvl-1/DeepSeek-V3.2/...          1   
                                   _logs/engine_vs_llm/dragon-lvl-2/DeepSeek-V3.2/...          4   
...                                                                                          ...   
qwen-3-235b-a22b-thinking-2507-low _logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a2...          0   
                                   _logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a2...          0   
qwen3.5-9b@q8_0                    _logs/rand_vs_llm/qwen-qwen3.5-9b/2026-03-03-14...          3   
ring-mini-2.0@q4_k_m               _logs/rand_vs_llm/ring-mini-2.0-q4_k_m/2025-10-...          0   
zai-glm-4.7                        _logs/rand_vs_llm/zai-glm-4.7/2026-01-27-21-01-20           2   

reason                                                                                 ERROR OCCURED  \
model                              run_path                                                            
DeepSeek-R1-0528                   _logs/engine_vs_llm/dragon-lvl-1/DeepSeek-R1-05...              5   
                                   _logs/engine_vs_llm/dragon-lvl-2/DeepSeek-R1-05...              4   
                                   _logs/rand_vs_llm/DeepSeek-R1-0528/2026-03-08-1...             24   
DeepSeek-V3.2                      _logs/engine_vs_llm/dragon-lvl-1/DeepSeek-V3.2/...              0   
                                   _logs/engine_vs_llm/dragon-lvl-2/DeepSeek-V3.2/...              1   
...                                                                                              ...   
qwen-3-235b-a22b-thinking-2507-low _logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a2...              0   
                                   _logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a2...              1   
qwen3.5-9b@q8_0                    _logs/rand_vs_llm/qwen-qwen3.5-9b/2026-03-03-14...              0   
ring-mini-2.0@q4_k_m               _logs/rand_vs_llm/ring-mini-2.0-q4_k_m/2025-10-...              0   
zai-glm-4.7                        _logs/rand_vs_llm/zai-glm-4.7/2026-01-27-21-01-20               0   

reason                                                                                 Fivefold repetition  \
model                              run_path                                                                  
DeepSeek-R1-0528                   _logs/engine_vs_llm/dragon-lvl-1/DeepSeek-R1-05...                    0   
                                   _logs/engine_vs_llm/dragon-lvl-2/DeepSeek-R1-05...                    0   
                                   _logs/rand_vs_llm/DeepSeek-R1-0528/2026-03-08-1...                    0   
DeepSeek-V3.2                      _logs/engine_vs_llm/dragon-lvl-1/DeepSeek-V3.2/...                    0   
                                   _logs/engine_vs_llm/dragon-lvl-2/DeepSeek-V3.2/...                    0   
...                                                                                                    ...   
qwen-3-235b-a22b-thinking-2507-low _logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a2...                    0   
                                   _logs/rand_vs_llm/_issues/ISSUES-qwen-3-235b-a2...                    0   
qwen3.5-9b@q8_0                    _logs/rand_vs_llm/qwen-qwen3.5-9b/2026-03-03-14...                    0   
ring-mini-2.0@q4_k_m               _logs/rand_vs_llm/ring-mini-2.0-q4_k_m/2025-10-...                    0   
zai-glm-4.7               

## Interpretation

- **Regenerate** `data_processing/elo_refined.csv` by running `data_processing/get_refined_csv.py` with `GAME_MODE=ELO` (see module docstring). The full Elo pipeline still reads legacy log directories unless you change that module; **this notebook** only inspects `rand_vs_llm` + `engine_vs_llm` for new-run QA.
- A model can appear in logs but **not** in `elo_refined.csv` if grouped `total_games` is below `FILTER_OUT_BELOW_N_RANDOM` / `FILTER_OUT_BELOW_N_MISC`, the label is in `FILTER_OUT_MODELS`, paths match `FILTER_OUT_PATH_KEYWORDS`, or Dragon ingestion skips a run (missing `_run.json` / opponent inference in `engine_vs_llm`). Use the **pending** lists and per-folder stats to see which case applies.
- **`TerminationReason`** examples: `"ERROR OCCURED"`, `"Too many wrong actions"`, `"Max turns in single dialog"`, checkmate/stalemate, etc. (`llm_chess.TerminationReason`).

In [8]:
# Smoke: imports + CSV reads + load_game_logs
assert Path(ELO_CSV).is_file()
assert Path(MODELS_METADATA_CSV).is_file()
assert isinstance(log_models, set)
print("smoke OK")

smoke OK
